# KisanSense: tomato disease model (Colab)

Trains two models on six tomato leaf classes from PlantVillage and compares them:
a small CNN baseline and MobileNetV2 transfer learning. Saves the selected model,
metrics and confusion matrices into your Drive copy of the project.

**Before you run**
1. Copy your whole `kisansense` project folder to the root of **My Drive** (the `data/` folder is not needed).
2. Runtime > Change runtime type > **T4 GPU**.
3. Have a Kaggle API token ready (Kaggle > Settings > API Tokens > Generate New Token). Best: store it in
   Colab Secrets under the name `KAGGLE_API_TOKEN` (key icon in the left sidebar, turn on Notebook access).
   Never paste a token into chat or share a screenshot of it.

Run the cells in order. The numbers this notebook prints are the real results to put in the README.

In [ ]:
# 1. Mount Drive, find the project, check the GPU
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT = '/content/drive/MyDrive/kisansense'
assert os.path.isdir(PROJECT + '/src/kisansense'), 'Copy the kisansense folder to My Drive first'
sys.path.insert(0, PROJECT + '/src')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| GPUs:', gpus)
assert gpus, 'No GPU: Runtime > Change runtime type > T4 GPU, then rerun'

In [ ]:
# 2. Kaggle credentials: read the API token from Colab Secrets, or ask for it (hidden input)
import os
!pip install -q --upgrade kaggle

try:
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    print('Using the KAGGLE_API_TOKEN secret')
except Exception:
    from getpass import getpass
    os.environ['KAGGLE_API_TOKEN'] = getpass('Paste your Kaggle API token (input is hidden): ')

# Quick check that the token works (should list a few PlantVillage datasets)
!kaggle datasets list -s plantvillage | head -5

In [ ]:
# 3. Download PlantVillage (about 2 GB) and extract ONLY the six tomato classes from the 'color' version
!mkdir -p /content/raw
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/raw

import glob
from kisansense.disease.data import extract_classes, TOMATO_CLASSES

zip_path = glob.glob('/content/raw/*.zip')[0]
counts = extract_classes(zip_path, '/content/data/tomato', TOMATO_CLASSES)
print(counts, '| total:', sum(counts.values()))
os.remove(zip_path)   # free disk space

In [ ]:
# 4. Look at the data: class balance and a few images
import matplotlib.pyplot as plt
from PIL import Image
from kisansense.disease.data import list_images

df = list_images('/content/data/tomato', TOMATO_CLASSES)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df['label'].str.replace('Tomato___', '').value_counts().plot.barh(ax=ax[0], title='Images per class')
ax[1].axis('off')
samples = df.groupby('label').head(1).path.tolist()
fig2, axes = plt.subplots(1, len(samples), figsize=(3 * len(samples), 3.4))
for a, p, c in zip(axes, samples, sorted(df.label.unique())):
    a.imshow(Image.open(p)); a.set_title(c.replace('Tomato___', ''), fontsize=9); a.axis('off')
plt.show()

In [ ]:
# 4b. Save EDA for the app's "Dataset and EDA" page: a sample grid and simple image statistics
import json
import numpy as np
import pandas as pd

EDA = PROJECT + '/docs/results/eda'
os.makedirs(EDA, exist_ok=True)
classes = sorted(df.label.unique())
short = lambda c: c.replace('Tomato___', '').replace('_', ' ')

# Sample grid: four random images per class
fig, axes = plt.subplots(len(classes), 4, figsize=(8, 2.1 * len(classes)))
for r, c in enumerate(classes):
    for k, pth in enumerate(df[df.label == c].path.sample(4, random_state=0)):
        ax = axes[r, k]
        ax.imshow(Image.open(pth).convert('RGB').resize((160, 160)))
        ax.axis('off')
        if k == 0:
            ax.set_title(short(c), fontsize=9, loc='left')
plt.tight_layout()
plt.savefig(EDA + '/disease_samples.png', dpi=110)
plt.show()

# Image statistics from up to 100 random images per class
rows = []
for c in classes:
    sub = df[df.label == c]
    for pth in sub.path.sample(min(100, len(sub)), random_state=0):
        im = Image.open(pth).convert('RGB')
        rows.append({'label': c, 'width': im.size[0], 'height': im.size[1],
                     'brightness': float(np.asarray(im, dtype=np.float32).mean() / 255)})
st = pd.DataFrame(rows)
stats = {
    'images_total': int(len(df)),
    'images_sampled': int(len(st)),
    'size_counts': [{'size': f'{w}x{h}', 'count': int(n)} for (w, h), n in st.groupby(['width', 'height']).size().sort_values(ascending=False).head(5).items()],
    'brightness_by_class': {short(c): {'mean': round(float(g.brightness.mean()), 4), 'std': round(float(g.brightness.std()), 4)} for c, g in st.groupby('label')},
}
with open(EDA + '/disease_eda.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

In [ ]:
# 5. Train both models and compare (about 10-20 minutes on a T4, my estimate)
from kisansense.disease.train import run, format_table

OUT = PROJECT + '/artifacts'
RES = PROJECT + '/docs/results'
result = run('/content/data/tomato', OUT, RES)

print(format_table(result['metrics']))
print()
print(result['metrics']['data'])

In [ ]:
# 6. Per-class F1 of the selected model on the test set
import pandas as pd
m = result['metrics']
best = m['selected_model']
print('Selected:', best)
pd.Series(m['results'][best]['test']['per_class_f1']).sort_values().round(3)

In [ ]:
# 7. Confusion matrix of the selected model
from IPython.display import Image as IPImage, display
display(IPImage(RES + f'/disease_confusion_{best}.png'))

In [ ]:
# 8. Grad-CAM: where does the model look? Shows mistakes first, then correct examples.
import numpy as np
from kisansense.disease.data import make_dataset
from kisansense.disease.gradcam import compute_gradcam, overlay

model, names = result['model'], result['class_names']
test_df, size = result['test_df'], result['image_size']

try:
    model.get_layer('backbone')
except ValueError:
    print('The selected model is the baseline CNN, which has no Grad-CAM. Skipping.')
else:
    pred = model.predict(make_dataset(test_df, names, size, 32), verbose=0).argmax(1)
    true = test_df['label'].map({c: i for i, c in enumerate(names)}).to_numpy()
    rng = np.random.default_rng(0)
    wrong = np.where(pred != true)[0]
    right = np.where(pred == true)[0]
    pick = list(wrong[:3]) + list(rng.choice(right, 6 - min(3, len(wrong)), replace=False))

    fig, axes = plt.subplots(2, len(pick), figsize=(3.2 * len(pick), 6.6))
    for col, i in enumerate(pick):
        img = Image.open(test_df.path[i]).convert('RGB').resize(size)
        arr = np.asarray(img, dtype=np.float32)
        cam, probs, _ = compute_gradcam(model, arr, int(pred[i]))
        axes[0, col].imshow(img); axes[0, col].axis('off')
        axes[0, col].set_title(f"true: {names[true[i]].replace('Tomato___','')}", fontsize=8)
        axes[1, col].imshow(overlay(arr.astype('uint8'), cam)); axes[1, col].axis('off')
        axes[1, col].set_title(f"pred: {names[pred[i]].replace('Tomato___','')} ({probs[pred[i]]:.2f})", fontsize=8,
                               color='green' if pred[i] == true[i] else 'red')
    plt.tight_layout()
    plt.savefig(RES + '/disease_gradcam_examples.png', dpi=130)
    plt.show()
    print(f'{len(wrong)} mistakes out of {len(test_df)} test images')

In [ ]:
# 8b. All the mistakes: which classes get confused, and what do the failures look like? (saved for the app)
import json
from collections import Counter

probs = model.predict(make_dataset(test_df, names, size, 32), verbose=0)
conf, pred = probs.max(1), probs.argmax(1)
true = test_df['label'].map({c: i for i, c in enumerate(names)}).to_numpy()
wrong = np.where(pred != true)[0]
short = lambda c: c.replace('Tomato___', '').replace('_', ' ')

pairs = Counter((short(names[true[i]]), short(names[pred[i]])) for i in wrong)
summary = {
    'mistakes': int(len(wrong)),
    'test_images': int(len(test_df)),
    'top_confusions': [{'true': t, 'predicted': p, 'count': n} for (t, p), n in pairs.most_common(8)],
    'mean_confidence_when_wrong': round(float(conf[wrong].mean()), 3),
    'mean_confidence_when_right': round(float(conf[pred == true].mean()), 3),
}
with open(EDA + '/disease_errors.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

# The 12 most confident mistakes
top = wrong[np.argsort(-conf[wrong])][:12]
fig, axes = plt.subplots(3, 4, figsize=(11, 9))
for ax, i in zip(axes.ravel(), top):
    ax.imshow(Image.open(test_df.path[i]).convert('RGB').resize((200, 200)))
    ax.set_title(f'true: {short(names[true[i]])}\npred: {short(names[pred[i]])} ({conf[i]:.2f})', fontsize=8, color='#b3261e')
    ax.axis('off')
for ax in axes.ravel()[len(top):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(EDA + '/disease_errors.png', dpi=110)
plt.show()

In [ ]:
# 9. What was saved to your Drive project folder
for folder in (OUT, RES):
    for f in sorted(os.listdir(folder)):
        print(folder.replace(PROJECT, '...'), '/', f)

## Next
Copy the printed comparison table (cell 5), the 'mistakes out of' line (cell 8) and a screenshot of the Grad-CAM
figure back into the chat. In your Drive, `artifacts/` holds the model and `docs/results/` the metrics and plots:
download both folders into your local project.

**Remember when you write the README:** PlantVillage photos are taken on plain backgrounds in a lab. High scores here
do not show the model works on photos taken in a field.

New cells 4b and 8b also save `docs/results/eda/` (sample grid, image statistics, mistake summary and gallery). Copy that whole folder next to your other results, then run `python scripts/make_eda.py` locally for the crop and price EDA.